## Middlewares

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

google_api_key = os.getenv("GOOGLE_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")

In [2]:
import langchain
langchain.__version__

'1.3.1'

In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

agent = create_agent(
    model="groq:llama-3.3-70b-versatile",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:llama-3.3-70b-versatile",
            trigger=("messages", 10),
            keep=("messages", 4)
        )
    ]
)

In [4]:
# Run with thread id
config={"configurable": {"thread_id": "test-1"}}

In [5]:
questions = [
    "What is Python?",
    "What is a variable?",
    "What is a function?",
    "What is a list in Python?",
    "What is a dictionary?",
    "What is a loop?",
    "What is an API?",
    "What is object-oriented programming?",
    "What is a database?",
    "What is machine learning?"
] 

In [6]:
for q in questions:
    response = agent.invoke(
        {"messages": [HumanMessage(content=q)]},
        config
    )

    print("Question:", q)
    print("Answer:", response["messages"][-1].content)
    print("Total Messages:", len(response["messages"]))
    print("-" * 50)

Question: What is Python?
Answer: **Python** is a high-level, interpreted programming language that is widely used for various purposes such as:

* Web development
* Data analysis and science
* Artificial intelligence and machine learning
* Automation
* Scientific computing
* Education

Python is known for its:

* **Easy-to-learn syntax**: Simple and intuitive syntax makes it a great language for beginners.
* **Flexibility**: Can be used for a wide range of applications, from simple scripts to complex systems.
* **Large community**: Active and supportive community, with numerous libraries, frameworks, and resources available.
* **Cross-platform**: Can run on multiple operating systems, including Windows, macOS, and Linux.

Some of the key features of Python include:

* **Dynamic typing**: Variables do not need to be declared before use.
* **Object-oriented programming**: Supports concepts such as classes, objects, and inheritance.
* **Modular design**: Code can be organized into module

KeyboardInterrupt: 

Based on  tokensize 

In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool

@tool
def search_hotels(city: str) -> str:
    """Return hotel names for a city. Use this tool only once."""
    
    return f"Hotels found in {city}: Grand Hotel, Royal Palace Hotel, City View Hotel, Ocean Breeze Resort, Comfort Inn Downtown"

agent = create_agent(
    model="groq:llama-3.1-8b-instant",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
    SummarizationMiddleware(
        model="groq:llama-3.1-8b-instant",
        trigger=("messages", 5),
        keep=("messages", 1)
    )
],
)

config={"configurable": {"thread_id": "test-1"}}

def count_tokens(messages):
    total_chars= sum(len(str(m.content)) for m in messages)
    return total_chars

In [6]:
system = SystemMessage(
    content="""
    You only have access to the search_hotels tool.

    Never call:
    - brave_search
    - web_search
    - google_search
    - any other tool

    If the user provides a country instead of a city,
    ask for a city name.
    """
)

In [8]:
cities= [
    "Paris",
    "London",
    "Tokyo",
    "Singapore",
    "Mumbai",
    "Delhi"
]

for city in cities:
    response = agent.invoke(
    {
        "messages": [
            system,
            HumanMessage(content=f"find hotels in {city}")
        ]
    },
    config=config
)
    tokens = count_tokens(response["messages"])
    print(f"{city} ~ {tokens} tokens, {len(response['messages'])} messages")
    print(f"{response['messages']}")
    # print(response["messages"][-1].content)

Paris ~ 1356 tokens, 5 messages
[HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nFind hotels in Paris and Australia, investigate an issue with the 'search_hotels' tool returning inconsistent results.\n\n## SUMMARY\nThe 'search_hotels' tool has been found to return the same list of hotels for all cities (Paris, Australia, Singapore, and Tokyo), indicating a potential issue with the tool. The accuracy of the results for Paris cannot be verified. To refine the list of hotels for Paris, follow-up questions or additional tools can be used to filter the results. The list of hotels in each city is the same: Grand Hotel, Royal Palace Hotel, City View Hotel, Ocean Breeze Resort, Comfort Inn Downtown.\n\n## ARTIFACTS\nList of hotels in Paris, Australia, Singapore, and Tokyo: Grand Hotel, Royal Palace Hotel, City View Hotel, Ocean Breeze Resort, Comfort Inn Downtown.\n\n## NEXT STEPS\nRefine the list of hotels for Paris by asking follow-up questions or u

BadRequestError: Error code: 400 - {'error': {'message': "tool call validation failed: attempted to call tool 'brave_search' which was not in request.tools", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=brave_search>{"query": "hotels in London"}</function>'}}